# Do quants actually use the formulas you were taught?

**Out of Sample** — [read the article](https://arthurnoo.pythonanywhere.com/formulas-in-practice)

This notebook reproduces every figure and every number in the article.

## Three ways to run it

| Where | What you need | Link |
| --- | --- | --- |
| **In your browser** | nothing at all | [Open in Colab](https://colab.research.google.com/github/Arthurnoo/out-of-sample/blob/main/notebooks/formulas-in-practice.ipynb) |
| **In VS Code** | VS Code installed | [Clone and open](vscode://vscode.git/clone?url=https://github.com/Arthurnoo/out-of-sample) |
| **Anywhere else** | Python 3.10+ | you already have the file |

The next cell installs whatever is missing, so in every case you can simply
**Run All** and watch. The first run downloads prices and takes a minute or two.

## How to read it

Each section matches a section of the article. Run a cell, look at what it
prints, then read the note underneath explaining what to make of it.

**Section 0 holds every parameter you might want to change.** Everything after
it is written so that changing those values changes the whole notebook. That is
the point: the numbers below are mine, and the interesting question is what
happens to them when you move something.

Your numbers may differ slightly from the article if you run this later. The
data keeps growing, which is itself worth noticing.

## If you find a mistake

Tell me. There is a form at the bottom of the article, and a wrong number
corrected in public is worth more than a right one nobody checked.

In [ ]:
# Installs anything missing, then does nothing on later runs.
# This is what makes "Run All" work in Colab, in VS Code, or on a bare machine.
import importlib, subprocess, sys

for package in ['yfinance', 'pandas', 'numpy', 'matplotlib', 'scipy']:
    try:
        importlib.import_module(package)
    except ImportError:
        print(f'installing {package} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', package], check=True)

print('ready')

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import datetime as dt

pd.set_option('display.width', 130)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True,
                     'grid.color': '#e4e9ee', 'axes.facecolor': 'white'})

## 0. The knobs

| Knob | What it does | Try |
| --- | --- | --- |
| `UNDERLYING` | the index or stock studied | `QQQ`, `IWM`, `AAPL` |
| `RATE` | the risk-free rate assumed | match the current T-bill |
| `DAY_RANGE` | which expiries to include | `(7, 800)` for the whole curve |
| `BAND` | how far from spot a strike may be | `0.02` is stricter and cleaner |
| `MIN_VOLUME` | contracts traded for a strike to count | **the one that decides data quality** |
| `STOCKS` | single names tested at the end | any liquid optionable ticker |

`MIN_VOLUME` is the important one. Lower it and you get more strikes and more
nonsense; raise it and you get fewer strikes and answers you can defend. Running
this during market hours changes everything for the better.

In [ ]:
# ─────────────────────────────────────────────────────────────────────
#  PARAMETERS — the only cell you need to edit
# ─────────────────────────────────────────────────────────────────────
UNDERLYING = 'SPY'
RATE       = 0.04          # flat assumption; bootstrapping the curve is better
DAY_RANGE  = (15, 400)     # expiries this many days out
BAND       = 0.05          # strikes within this fraction of spot
MIN_VOLUME = 1             # contracts traded, both call and put

STOCKS = ['AAPL','MSFT','NVDA','TSLA','AMZN','META','KO','JNJ','XOM','PG']
STOCK_MIN_VOLUME = 50      # stricter for single names, which are thinner

## 1. The move that makes this possible

Put-call parity, for the same strike and expiry:

```
C − P = (F − K) · e^(−rT)
```

This is **not a model**. No volatility, no assumption about dividends, no view
on the underlying. It is an arbitrage identity: buy a call, sell a put, and you
have synthesised the forward.

So invert it and read the market's own forward straight out of the prices:

```
F = K + (C − P) · e^(rT)
```

In [ ]:
def implied_forward(ticker, expiry, r=RATE, band=BAND, min_vol=MIN_VOLUME):
    tk = yf.Ticker(ticker)
    S = float(tk.history(period='1d')['Close'].iloc[-1])
    T = (dt.date.fromisoformat(expiry) - dt.date.today()).days / 365.0
    ch = tk.option_chain(expiry)
    m = ch.calls.merge(ch.puts, on='strike', suffixes=('_c', '_p'))
    for side in ('c', 'p'):
        m[f'mid_{side}'] = np.where(
            (m[f'bid_{side}'] > 0) & (m[f'ask_{side}'] > 0),
            (m[f'bid_{side}'] + m[f'ask_{side}']) / 2, m[f'lastPrice_{side}'])
    sel = m[(m.strike > S*(1-band)) & (m.strike < S*(1+band))
            & (m.volume_c.fillna(0) >= min_vol) & (m.volume_p.fillna(0) >= min_vol)
            & (m.mid_c > 0) & (m.mid_p > 0)].copy()
    if len(sel) < 3:
        return None
    sel['F'] = sel.strike + (sel.mid_c - sel.mid_p) * np.exp(r * T)
    F = float(sel.F.median())          # median: one bad quote cannot move it
    return {'expiry': expiry, 'T': T, 'spot': S, 'n_strikes': len(sel),
            'dispersion': float(sel.F.std()),
            'F_market': F, 'F_textbook': S * np.exp(r * T),
            'implied_yield': r - np.log(F / S) / T}

today = dt.date.today()
tk = yf.Ticker(UNDERLYING)
expiries = [e for e in tk.options
            if DAY_RANGE[0] <= (dt.date.fromisoformat(e) - today).days <= DAY_RANGE[1]]
print(f"{len(expiries)} expiries in range, from {expiries[0]} to {expiries[-1]}")

## 2. The textbook against the market

In [ ]:
rows = [o for e in expiries if (o := implied_forward(UNDERLYING, e))]
fw = pd.DataFrame(rows)
fw['gap_points'] = fw.F_textbook - fw.F_market
fw['days'] = fw['T'] * 365
S = float(fw.spot.iloc[0])

print(f"spot {S:.2f}\n")
print(fw[['expiry','days','n_strikes','F_market','F_textbook','gap_points','implied_yield']]
      .to_string(index=False, float_format=lambda v: f'{v:12.4f}'))

fig, (a, b) = plt.subplots(2, 1, figsize=(7.4, 5.2), sharex=True,
                           gridspec_kw={'height_ratios':[2, 1.2]})
a.plot(fw.days, fw.F_textbook, marker='o', ms=4, color='#c83c50', label=r'textbook $Se^{rT}$')
a.plot(fw.days, fw.F_market,   marker='o', ms=4, color='#0e8f5f', label='implied by options')
a.axhline(S, color='#8b98a6', ls=':', label=f'spot {S:.0f}')
a.legend(); a.set_ylabel('forward price')
b.bar(fw.days, fw.gap_points, width=max(fw.days.max()/40, 2), color='#2d6fa8')
b.axhline(0, color='k', lw=1); b.set_xlabel('days to expiry'); b.set_ylabel('gap')
plt.show()

## 3. What the gap contains

A practitioner writes the formula with the terms a course drops:

```
F = S · e^((r − q − b)T)
```

`q` is the dividend yield, `b` the cost of borrowing the asset. The textbook
version sets both to zero, so the gap measures exactly `q + b`:

```
q + b = r − ln(F/S) / T
```

In [ ]:
reported = (tk.info.get('trailingAnnualDividendYield') or tk.info.get('yield') or 0) * 100
print(f"implied carry, averaged over the curve : {fw.implied_yield.mean()*100:.2f}% a year")
print(f"dividend yield as published            : {reported:.2f}% a year")
print(f"difference                             : {fw.implied_yield.mean()*100 - reported:+.2f} point")
print("\nOn a broad, easy-to-borrow ETF the borrow term is near zero,")
print("so almost the entire disagreement with the textbook IS the dividend.")

## 4. A third opinion: the quoted future

In [ ]:
try:
    spx = float(yf.Ticker('^GSPC').history(period='1d')['Close'].iloc[-1])
    es  = float(yf.Ticker('ES=F').history(period='1d')['Close'].iloc[-1])
    print(f"S&P 500 spot : {spx:,.2f}")
    print(f"E-mini quote : {es:,.2f}")
    print(f"basis        : {es-spx:+.2f} points   ({(es/spx-1)*100:+.3f}%)")
    print("\nSame ingredients: financing to expiry, minus the dividends paid before then.")
except Exception as e:
    print('futures unavailable:', e)

## 5. Single stocks, and knowing when to refuse

Same extraction, stricter filter. Names without enough liquid strikes are
**dropped rather than guessed**.

In [ ]:
ok, rejected = [], []
for s in STOCKS:
    try:
        t2 = yf.Ticker(s)
        ex = [e for e in t2.options if 25 <= (dt.date.fromisoformat(e) - today).days <= 120]
        if not ex:
            rejected.append((s, 'no expiry in range')); continue
        o = implied_forward(s, ex[len(ex)//2], band=0.03, min_vol=STOCK_MIN_VOLUME)
        if not o:
            rejected.append((s, 'fewer than 3 liquid strikes')); continue
        rep = (t2.info.get('trailingAnnualDividendYield') or 0) * 100
        ok.append({'ticker': s, 'strikes': o['n_strikes'], 'dispersion': o['dispersion'],
                   'implied %': o['implied_yield']*100, 'published div %': rep,
                   'difference': o['implied_yield']*100 - rep})
    except Exception as e:
        rejected.append((s, str(e)[:30]))

st = pd.DataFrame(ok)
print(st.to_string(index=False, float_format=lambda v: f'{v:14.3f}'))
print(f"\nmedian absolute difference: {st.difference.abs().median():.2f} point a year")
print(f"dropped for illiquidity   : {', '.join(f'{a} ({b})' for a, b in rejected)}")
print("\nThat is not a failure of the formula. You cannot read a market price")
print("where there is no market, and refusing to answer is part of the job.")

## 6. Things worth trying yourself

1. **Run this during market hours.** The dispersion across strikes should
   collapse and the single-stock numbers should tighten considerably.
2. **Raise `MIN_VOLUME` to 200.** How many names survive? How much better is the
   agreement for those that do?
3. **Change `RATE` to 3% and to 5%.** Every implied yield shifts by the same
   amount. That tells you how much of your answer is the rate you assumed, which
   is the argument for bootstrapping it from the Treasury curve.
4. **Find a hard-to-borrow stock** (a heavily shorted small cap) and run it. The
   borrow term stops being negligible and the implied carry goes well above the
   dividend. That is `b` becoming visible.

Found something I got wrong? There is a form at the bottom of the article.